In [1]:
import gc
import random
import time

import torch

from edge_detector.scripts.benchmark import (
    benchmark_module,
    configure_cpu,
)
from edge_detector.scripts.evaluation import (
    load_checkpoint_model,
    prepare_for_inference,
)


INPUT_SIZE = 640
THREADS = 4
WARMUP = 1
RUNS = 3
RANDOM_SEED = 42
COOLDOWN_SECONDS = 0

configure_cpu(THREADS)

c:\etu\mag_diploma\diploma_code\rpi4-edge-detector\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\etu\mag_diploma\diploma_code\rpi4-edge-detector\third_party\YOLOv6\yolov6\utils\general.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg


In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
WEIGHTS_DIR = PROJECT_ROOT / "weights"

DETECTORS = {
    "InvertedResidual": WEIGHTS_DIR / "state_dict" / "custom_shufflenet.pt",
    "MLPBlock + PartialConv": WEIGHTS_DIR / "state_dict" / "custom_fasternet.pt",
    "RepVGG": WEIGHTS_DIR / "state_dict" / "custom_repvgg.pt",
    "C2f": WEIGHTS_DIR / "state_dict" / "custom_c2f.pt",
    "ShuffleNetV2 x0.5": WEIGHTS_DIR / "state_dict" / "baseline_shufflenetv2_x0_5.pt",
    "FasterNet T0": WEIGHTS_DIR / "state_dict" / "baseline_fasternet_t0.pt",
    "YOLOv8n": WEIGHTS_DIR / "state_dict" / "baseline_yolov8n.pt",
    "YOLOv6n": WEIGHTS_DIR / "state_dict" / "baseline_yolov6n.pt",
}

In [3]:
x = torch.randn(
    1,
    3,
    INPUT_SIZE,
    INPUT_SIZE,
    dtype=torch.float32,
)

detector_names = list(DETECTORS)
random.Random(RANDOM_SEED).shuffle(detector_names)

print(f"Models: {len(detector_names)}")

Models: 8


In [4]:
def benchmark_detector(
    detector_name: str,
    checkpoint_path: str,
) -> dict:
    model = prepare_for_inference(
        load_checkpoint_model(
            checkpoint_path,
            device="cpu",
        )
    )

    result = benchmark_module(
        model,
        x,
        warmup=WARMUP,
        runs=RUNS,
    )

    return {
        "model": detector_name,
        "checkpoint": checkpoint_path,
        **result.as_dict(),
    }

In [5]:
results = []

for index, detector_name in enumerate(detector_names, start=1):
    checkpoint_path = DETECTORS[detector_name]

    result = benchmark_detector(
        detector_name=detector_name,
        checkpoint_path=checkpoint_path,
    )
    results.append(result)

    print(
        f"[{index:2d}/{len(detector_names)}] "
        f"{result['model']:24s} | "
        f"params={result['params'] / 1e6:7.3f} M | "
        f"median={result['median_ms']:8.2f} ms | "
        f"mean={result['mean_ms']:8.2f} ms | "
        f"min={result['min_ms']:8.2f} ms | "
        f"max={result['max_ms']:8.2f} ms"
    )

    gc.collect()

    if COOLDOWN_SECONDS and index < len(detector_names):
        time.sleep(COOLDOWN_SECONDS)

[ 1/8] C2f                      | params=  1.540 M | median=   86.83 ms | mean=  104.61 ms | min=   79.67 ms | max=  147.32 ms
[ 2/8] ShuffleNetV2 x0.5        | params=  0.490 M | median=   89.60 ms | mean=   97.84 ms | min=   88.27 ms | max=  115.65 ms
[ 3/8] YOLOv8n                  | params=  1.704 M | median=  153.11 ms | mean=  155.49 ms | min=  142.36 ms | max=  170.99 ms
[ 4/8] YOLOv6n                  | params=  3.565 M | median=   95.63 ms | mean=   95.64 ms | min=   93.31 ms | max=   97.99 ms
[ 5/8] RepVGG                   | params=  1.793 M | median=   59.54 ms | mean=   58.30 ms | min=   55.45 ms | max=   59.91 ms
[ 6/8] FasterNet T0             | params=  2.736 M | median=  111.86 ms | mean=  116.99 ms | min=  111.28 ms | max=  127.84 ms
[ 7/8] InvertedResidual         | params=  0.883 M | median=   56.01 ms | mean=   55.31 ms | min=   51.62 ms | max=   58.30 ms
[ 8/8] MLPBlock + PartialConv   | params=  1.318 M | median=   76.68 ms | mean=   72.83 ms | min=   64.42 ms | 

In [6]:
sorted_results = sorted(
    results,
    key=lambda result: result["median_ms"],
)

header = (
    f"{'Detector':24s} "
    f"{'Params, M':>10s} "
    f"{'Median, ms':>12s} "
    f"{'Mean, ms':>10s} "
    f"{'Min, ms':>10s} "
    f"{'Max, ms':>10s}"
)

print(header)
print("-" * len(header))

for result in sorted_results:
    print(
        f"{result['model']:24s} "
        f"{result['params'] / 1e6:10.3f} "
        f"{result['median_ms']:12.2f} "
        f"{result['mean_ms']:10.2f} "
        f"{result['min_ms']:10.2f} "
        f"{result['max_ms']:10.2f}"
    )

Detector                  Params, M   Median, ms   Mean, ms    Min, ms    Max, ms
---------------------------------------------------------------------------------
InvertedResidual              0.883        56.01      55.31      51.62      58.30
RepVGG                        1.793        59.54      58.30      55.45      59.91
MLPBlock + PartialConv        1.318        76.68      72.83      64.42      77.40
C2f                           1.540        86.83     104.61      79.67     147.32
ShuffleNetV2 x0.5             0.490        89.60      97.84      88.27     115.65
YOLOv6n                       3.565        95.63      95.64      93.31      97.99
FasterNet T0                  2.736       111.86     116.99     111.28     127.84
YOLOv8n                       1.704       153.11     155.49     142.36     170.99
